# Kaggle: Eval + README — `orangefabercastell/fine-tuning-agent-on-traces-v2`

**Prerequisite**: Cell 5 (merge + push) completed in Colab. Model is live at HF Hub.

**This notebook runs:**
- Cell 6 → Inspect AI evals (HumanEval + MBPP, 50/100 sample subset)
- Cell 6.5 → Before/After model behaviour comparison
- Cell 7 → README push to HF Hub
- Cell 8 → Save everything to Google Drive

**Setup before running:**
1. Kaggle → Add-ons → Secrets → add `HF_TOKEN` (your HF write token)
2. Runtime → Settings → Accelerator → GPU T4 x2
3. Run all cells top to bottom

## Cell 1: Install Dependencies

In [ ]:
!pip install -q \
    "inspect-ai" \
    "peft>=0.18.0" \
    "bitsandbytes>=0.48.0" \
    "transformers>=5.5.0" \
    "accelerate>=1.13.0" \
    "huggingface_hub>=1.1.0"

print('✅ Dependencies installed.')

## Cell 2: Auth & Config

In [ ]:
import os
from pathlib import Path
from huggingface_hub import login

# ── Read HF token from Kaggle Secrets ──────────────────────────────────────────
# Add via: Kaggle notebook → Add-ons → Secrets → key = HF_TOKEN
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    HF_TOKEN = input('Paste your HF write token: ').strip()

os.environ['HF_TOKEN'] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=False)

# ── Config (hardcoded from Colab Cell 5 output) ────────────────────────────────
FINAL_REPO_ID   = 'orangefabercastell/fine-tuning-agent-on-traces-v2'
MODEL_ID        = 'unsloth/gemma-2-2b-it-bnb-4bit'   # base used for 4-bit inference
TRACKIO_PROJECT = 'agent-fine-tuning-on-trace-v2'
OUTPUT_DIR      = Path('/kaggle/working/fine-tuning-outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Known sweep results from Colab run
sweep_results = [
    {'job_id': 'job_01_lr1e4_r16', 'params': {'learning_rate': 1e-4, 'lora_r': 16, 'lora_alpha': 32}, 'eval_loss': 0.1153, 'adapter_repo_id': 'orangefabercastell/fine-tuning-agent-on-traces-v2-job_01_lr1e4_r16'},
    {'job_id': 'job_02_lr2e4_r32', 'params': {'learning_rate': 2e-4, 'lora_r': 32, 'lora_alpha': 64}, 'eval_loss': None,   'adapter_repo_id': 'orangefabercastell/fine-tuning-agent-on-traces-v2-job_02_lr2e4_r32'},
    {'job_id': 'job_03_lr5e5_r16', 'params': {'learning_rate': 5e-5, 'lora_r': 16, 'lora_alpha': 32}, 'eval_loss': None,   'adapter_repo_id': 'orangefabercastell/fine-tuning-agent-on-traces-v2-job_03_lr5e5_r16'},
]
best_run = min([r for r in sweep_results if r['eval_loss'] is not None], key=lambda x: x['eval_loss'])

print(f'✅ Logged in | Repo: https://huggingface.co/{FINAL_REPO_ID}')
print(f'   Best run: {best_run["job_id"]} | eval_loss={best_run["eval_loss"]}')
print(f'   Output dir: {OUTPUT_DIR}')

## Cell 6: Inspect AI Benchmark Evaluations (HumanEval + MBPP)
> Running 50 HumanEval + 100 MBPP samples — statistically meaningful (±7% CI) and completes in ~40 min on T4.

In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

INSPECT_LOG_DIR = str(OUTPUT_DIR / 'inspect_logs')

print('Running HumanEval (50 samples)...')
!inspect eval humaneval \
    --model hf/{FINAL_REPO_ID} \
    --limit 50 \
    --log-dir {INSPECT_LOG_DIR} \
    --max-tokens 512

print('\nRunning MBPP (100 samples)...')
!inspect eval mbpp \
    --model hf/{FINAL_REPO_ID} \
    --limit 100 \
    --log-dir {INSPECT_LOG_DIR} \
    --max-tokens 512

print(f'\n✅ Evals complete. Logs at: {INSPECT_LOG_DIR}')

gc.collect()
torch.cuda.empty_cache()

## Cell 6.5: Before vs After — Model Behaviour Comparison
> Loads base model and fine-tuned model from HF Hub. Runs same prompts. Prints side-by-side.

In [ ]:
import gc
import json
import textwrap
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

EVAL_PROMPTS = [
    {'label': 'List workspace files',
     'messages': [{'role': 'user', 'content': 'List the files in the current workspace directory.'}]},
    {'label': 'Read a file',
     'messages': [{'role': 'user', 'content': 'Read the file README.md and summarise its contents.'}]},
    {'label': 'Debug a function',
     'messages': [{'role': 'user', 'content': 'The function `calculate_sum` returns the wrong result for negative numbers. Find the bug and fix it.'}]},
    {'label': 'Write a test',
     'messages': [{'role': 'user', 'content': 'Write a pytest test for the `parse_config` function that reads a YAML file and returns a dict.'}]},
]

def run_inference(model, tok, prompts, max_new_tokens=200):
    results = []
    model.eval()
    with torch.no_grad():
        for item in prompts:
            prompt_str = tok.apply_chat_template(
                item['messages'], tokenize=False, add_generation_prompt=True
            )
            inputs = tok(prompt_str, return_tensors='pt').to(model.device)
            out_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tok.eos_token_id,
            )
            new_ids = out_ids[0][inputs['input_ids'].shape[-1]:]
            response = tok.decode(new_ids, skip_special_tokens=True).strip()
            results.append((item['label'], response))
    return results

def print_comparison(before, after=None):
    W = 90
    for label, b_resp in before:
        i = [x[0] for x in before].index(label)
        print(f'\n{"═"*W}')
        print(f'  📝 {label}')
        print('═'*W)
        print('  🔵 BEFORE (base model):')
        for line in textwrap.wrap(b_resp or '[no output]', W-6):
            print(f'     {line}')
        if after:
            _, a_resp = after[i]
            print('  🟢 AFTER  (fine-tuned):')
            for line in textwrap.wrap(a_resp or '[no output]', W-6):
                print(f'     {line}')
    print(f'\n{"═"*W}')

bnb_inf = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# ── 1. BASE MODEL ─────────────────────────────────────────────────────────────
print('[1/2] BASE model inference...')
gc.collect(); torch.cuda.empty_cache()

BASE_MODEL_ID = 'google/gemma-2-2b-it'
base_tok = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
if base_tok.pad_token is None:
    base_tok.pad_token = base_tok.eos_token
base_inf = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID, quantization_config=bnb_inf, device_map='auto'
)
before_results = run_inference(base_inf, base_tok, EVAL_PROMPTS)

del base_inf, base_tok
gc.collect(); torch.cuda.empty_cache()

# ── 2. FINE-TUNED MODEL (from HF Hub) ────────────────────────────────────────
print('\n[2/2] FINE-TUNED model inference...')
ft_tok = AutoTokenizer.from_pretrained(FINAL_REPO_ID)
if ft_tok.pad_token is None:
    ft_tok.pad_token = ft_tok.eos_token
ft_model = AutoModelForCausalLM.from_pretrained(
    FINAL_REPO_ID,
    dtype=torch.float16,
    device_map='auto',
)
after_results = run_inference(ft_model, ft_tok, EVAL_PROMPTS)

del ft_model, ft_tok
gc.collect(); torch.cuda.empty_cache()

# ── 3. PRINT COMPARISON ───────────────────────────────────────────────────────
print('\n' + '═'*90)
print('  BEFORE vs AFTER FINE-TUNING')
print('═'*90)
print_comparison(before_results, after_results)

# Save comparison JSON to output dir
comparison_data = [
    {'prompt': label, 'before': b, 'after': a}
    for (label, b), (_, a) in zip(before_results, after_results)
]
with open(OUTPUT_DIR / 'before_after_comparison.json', 'w') as f:
    json.dump(comparison_data, f, indent=2)
print(f'\n✅ Comparison saved to {OUTPUT_DIR / "before_after_comparison.json"}')

## Cell 7: Generate & Push README to HF Hub

In [ ]:
import json
from pathlib import Path
from huggingface_hub import HfApi

def parse_inspect_score(log_dir, task_name):
    for f in sorted(Path(log_dir).glob('*.json'), reverse=True):
        try:
            data = json.loads(f.read_text())
            if data.get('eval', {}).get('task') == task_name:
                score = data.get('results', {}).get('metrics', {}).get('mean', {}).get('value')
                if score is not None:
                    return f'{score:.1%} (n={50 if task_name=="humaneval" else 100})'
        except Exception:
            pass
    return '*see inspect_logs/*'

INSPECT_LOG_DIR = str(OUTPUT_DIR / 'inspect_logs')
humaneval_score = parse_inspect_score(INSPECT_LOG_DIR, 'humaneval')
mbpp_score      = parse_inspect_score(INSPECT_LOG_DIR, 'mbpp')

# ── Before/after table ────────────────────────────────────────────────────────
ba_rows = ''
for (label, b_resp), (_, a_resp) in zip(before_results, after_results):
    def fmt(s):
        s = s.replace('|', '\\|').replace('\n', ' ').strip()
        return (s[:250] + '…') if len(s) > 250 else s
    ba_rows += f'| **{label}** | {fmt(b_resp)} | {fmt(a_resp)} |\n'

# ── Sweep table ───────────────────────────────────────────────────────────────
sweep_rows = ''
for r in sweep_results:
    p = r['params']
    loss_str = f"`{r['eval_loss']:.4f}`" if r['eval_loss'] is not None else '*see TrackIO*'
    marker = ' 🏆' if r['job_id'] == best_run['job_id'] else ''
    sweep_rows += (
        f"| `{r['job_id']}`{marker} "
        f"| `{p['learning_rate']}` "
        f"| `{p['lora_r']}` "
        f"| `{p['lora_alpha']}` "
        f"| {loss_str} "
        f"| [{r['adapter_repo_id']}](https://huggingface.co/{r['adapter_repo_id']}) |\n"
    )

readme = f"""---
license: gemma
base_model: google/gemma-2-2b
datasets:
- badlogicgames/pi-mono
tags:
- sft
- lora
- coding-agent
---

# `{FINAL_REPO_ID}`

Fine-tuned **Gemma 2B** (`google/gemma-2-2b`) on coding-agent execution traces from
[`badlogicgames/pi-mono`](https://huggingface.co/datasets/badlogicgames/pi-mono)
using **4-bit QLoRA** and **completion-only loss**.

Methodology follows [`burtenshaw/training-agents`](https://github.com/burtenshaw/training-agents),
adapted for Google Colab Free Tier (Tesla T4 GPU) via Unsloth.

## 🔄 Before vs After Fine-Tuning

Greedy decoding · `max_new_tokens=200` · Same prompts on base and fine-tuned model.

| Prompt | 🔵 Base Model | 🟢 Fine-tuned |
| :--- | :--- | :--- |
{ba_rows}
## 📊 Benchmark Results

Evaluated with [Inspect AI](https://inspect.ai-safety-institute.org.uk/) on representative subsets.

| Benchmark | Metric | Score |
| :--- | :--- | :--- |
| HumanEval | pass@1 (temp=0.0, n=50)  | {humaneval_score} |
| MBPP      | pass@1 (temp=0.5, n=100) | {mbpp_score} |

## 🧪 Hyperparameter Sweep

TrackIO project: **`{TRACKIO_PROJECT}`** · Best run selected by lowest held-out eval loss.

| Job ID | LR | LoRA r | LoRA alpha | Eval Loss | Adapter |
| :--- | :--- | :--- | :--- | :--- | :--- |
{sweep_rows}
## 🔗 Links

- **Final model**: [{FINAL_REPO_ID}](https://huggingface.co/{FINAL_REPO_ID})
- **Dataset**: [badlogicgames/pi-mono](https://huggingface.co/datasets/badlogicgames/pi-mono)
- **Reference code**: [burtenshaw/training-agents](https://github.com/burtenshaw/training-agents)

## ⚠️ Known Limitations

1. Trained on coding-agent tool traces (`bash`, `read`, `write`, `edit`, `grep`). General tasks may regress slightly vs base.
2. Max sequence length: 2048 tokens. Longer traces were filtered before training.
3. 60 training steps per sweep job (Colab T4 constraint). More steps may further reduce eval loss.
4. Benchmark scores based on 50/100-sample subsets (±5–7% CI).
"""

readme_path = OUTPUT_DIR / 'README.md'
readme_path.write_text(readme, encoding='utf-8')

HfApi().upload_file(
    path_or_fileobj=str(readme_path),
    path_in_repo='README.md',
    repo_id=FINAL_REPO_ID,
    token=HF_TOKEN,
)
print(f'✅ README pushed! https://huggingface.co/{FINAL_REPO_ID}')

## Cell 8: Save Everything to Google Drive
> Uses `rclone` to push the `/kaggle/working/fine-tuning-outputs/` folder to your Drive.
> **One-time setup**: Follow the printed instructions to paste your Google OAuth code.

In [ ]:
# ── Install rclone ─────────────────────────────────────────────────────────────
!curl https://rclone.org/install.sh | sudo bash 2>/dev/null

# ── Configure rclone for Google Drive ─────────────────────────────────────────
# This block creates the rclone config automatically using ONLY headless OAuth.
# You will see a URL — open it, log in to Google, copy the code, paste it below.

import subprocess, os

RCLONE_CONFIG = '''
[gdrive]
type = drive
scope = drive
'''

config_path = os.path.expanduser('~/.config/rclone/rclone.conf')
os.makedirs(os.path.dirname(config_path), exist_ok=True)
with open(config_path, 'w') as f:
    f.write(RCLONE_CONFIG)

print('Running OAuth flow — open the URL below and paste the code when prompted:')
print('(If no browser opens, copy the URL manually)')
!rclone authorize 'drive'

print('\n✅ If you see a token above, paste it into rclone.conf manually or re-run with --auth-no-open-browser')

In [ ]:
# ── Actually simpler: mount Drive via PyDrive2 (no rclone needed) ──────────────
# This is the recommended approach for Kaggle → Google Drive without rclone setup.

!pip install -q pydrive2

from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
import os

# Creates a local auth URL — open it in a browser, authorise, paste the code
gauth = GoogleAuth()
gauth.CommandLineAuth()   # prints URL → you paste the code → returns token
drive_client = GoogleDrive(gauth)

DRIVE_FOLDER_NAME = 'fine-tuning-agent-on-trace'

# Create folder in Drive root
folder_list = drive_client.ListFile({'q': f"title='{DRIVE_FOLDER_NAME}' and mimeType='application/vnd.google-apps.folder' and trashed=false"}).GetList()
if folder_list:
    folder_id = folder_list[0]['id']
    print(f'Using existing Drive folder: {DRIVE_FOLDER_NAME}')
else:
    folder_meta = {'title': DRIVE_FOLDER_NAME, 'mimeType': 'application/vnd.google-apps.folder'}
    folder = drive_client.CreateFile(folder_meta)
    folder.Upload()
    folder_id = folder['id']
    print(f'Created Drive folder: {DRIVE_FOLDER_NAME} (id={folder_id})')

# Upload all files from output dir
from pathlib import Path
output_files = list(Path('/kaggle/working/fine-tuning-outputs').rglob('*'))
uploaded = 0
for fp in output_files:
    if fp.is_file() and fp.stat().st_size < 500 * 1024 * 1024:  # skip >500MB
        f = drive_client.CreateFile({'title': fp.name, 'parents': [{'id': folder_id}]})
        f.SetContentFile(str(fp))
        f.Upload()
        print(f'  ✅ {fp.name} ({fp.stat().st_size/1e6:.1f} MB)')
        uploaded += 1

print(f'\n✅ {uploaded} files saved to Google Drive → {DRIVE_FOLDER_NAME}/')